# Kaggle Solar Filament Segmentation Challenge 2026
## Complete End-to-End Training, Evaluation & Submission Pipeline
**Team:** Dr. T Solar Filament Lab | Zenieverse  
**Competition:** https://www.kaggle.com/competitions/filament-segmentation-2026  
**Target Metric:** Panoptic Quality ($PQ = SQ \times RQ$ with bipartite matching at $\text{IoU} > 0.5$)  
**Resolution:** Full-disk 2048 × 2048 Balmer H-alpha ($656.3\text{ nm}$) imagery

In [ ]:
# 1. Environment & Package Setup
!pip install -q albumentations timm opencv-python-headless scikit-image pycocotools astropy torch torchvision pandas

import os
import random
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from skimage.feature import hessian_matrix, hessian_matrix_eigvals
from skimage.segmentation import watershed
from skimage.measure import label, regionprops

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f'Execution Device: {device}')

### 2. Physical Preprocessing: Eddington Limb Darkening Inversion & Hessian Tubeness
The photospheric intensity profile follows Eddington's linear law:
$$I(r) = I_0 \cdot [1 - u \cdot (1 - \sqrt{1 - (r/R_\odot)^2})]$$
For Balmer H-alpha ($656.3\text{ nm}$), $u \approx 0.62$.

In [ ]:
def correct_limb_darkening(image_gray: np.ndarray, center=(1024, 1024), radius=940, u=0.62) -> np.ndarray:
    h, w = image_gray.shape
    y, x = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((x - center[0])**2 + (y - center[1])**2)
    normalized_r = np.clip(dist_from_center / radius, 0.0, 1.0)
    mu = np.sqrt(np.maximum(0.0, 1.0 - normalized_r**2))
    darkening_factor = np.clip(1.0 - u * (1.0 - mu), 0.25, 1.0)
    corrected = image_gray.astype(np.float32) / darkening_factor
    disk_mask = dist_from_center <= (0.98 * radius)
    corrected[~disk_mask] = 0
    return np.clip(corrected, 0, 255).astype(np.uint8)

def compute_hessian_tubeness(image: np.ndarray, sigmas=(1.5, 3.0, 4.5, 7.0), valid_mask=None) -> np.ndarray:
    inverted = 255.0 - image.astype(np.float32)
    max_response = np.zeros_like(inverted)
    for sigma in sigmas:
        H_elems = hessian_matrix(inverted, sigma=sigma, order='rc', use_gaussian_derivatives=False)
        lambda1, lambda2 = hessian_matrix_eigvals(H_elems)
        tubeness = (float(sigma) ** 2) * np.maximum(0.0, -lambda2)
        max_response = np.maximum(max_response, tubeness)
    if valid_mask is not None:
        max_response[~valid_mask] = 0.0
    peak = float(np.max(max_response))
    if peak < 3.0:
        return np.zeros(image.shape, dtype=np.uint8)
    norm_response = (max_response / peak * 255.0).astype(np.uint8)
    return norm_response

def segment_filament_instances(image_gray: np.ndarray, center=(1024, 1024), radius=940) -> list:
    corr = correct_limb_darkening(image_gray, center=center, radius=radius)
    h, w = image_gray.shape
    y, x = np.ogrid[:h, :w]
    dist = np.sqrt((x - center[0])**2 + (y - center[1])**2)
    inner_disk = dist <= (0.95 * radius)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(corr)
    disk_pixels = enhanced[inner_disk]
    enhanced_for_hessian = enhanced.copy()
    if disk_pixels.size > 0:
        enhanced_for_hessian[~inner_disk] = int(np.median(disk_pixels))
    tubeness = compute_hessian_tubeness(enhanced_for_hessian, valid_mask=inner_disk)
    if not np.any(tubeness):
        return []
    thresh = cv2.adaptiveThreshold(tubeness, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, -5)
    thresh[~inner_disk] = 0
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
    cleaned[~inner_disk] = 0
    return separate_instances(cleaned)


### 3. PyTorch Dataset & Augmentation Pipeline

In [ ]:
class SolarFilamentDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, transform=None, is_train=True):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            image = np.zeros((2048, 2048), dtype=np.uint8)
        
        # Physical preprocessing
        image = correct_limb_darkening(image)
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        image = clahe.apply(image)
        
        if self.mask_paths and idx < len(self.mask_paths) and self.mask_paths[idx]:
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros_like(image, dtype=np.float32)

        # Normalize to [0, 1]
        img_tensor = torch.tensor(image / 255.0, dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return img_tensor, mask_tensor, os.path.basename(img_path)

### 4. Neural Network Architecture: Solar U-Net with ResNet Backbone & Attention

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class SolarFilamentUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.inc = DoubleConv(in_channels, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv_up1 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv_up2 = DoubleConv(128, 64)
        self.up3 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv_up3 = DoubleConv(64, 32)
        self.outc = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x = self.up1(x4)
        x = self.conv_up1(torch.cat([x, x3], dim=1))
        x = self.up2(x)
        x = self.conv_up2(torch.cat([x, x2], dim=1))
        x = self.up3(x)
        x = self.conv_up3(torch.cat([x, x1], dim=1))
        return self.outc(x)

### 5. Composite Loss: Soft Dice Loss + Focal Loss

In [ ]:
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        intersection = (probs_flat * targets_flat).sum()
        dice = (2. * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        return 1.0 - dice

class FilamentLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = SoftDiceLoss()
    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.dice(logits, targets)

### 6. Official Panoptic Quality ($PQ = SQ \times RQ$) Evaluation Function

In [ ]:
def calculate_panoptic_quality(pred_instances, gt_instances):
    if len(pred_instances) == 0 and len(gt_instances) == 0:
        return {'pq': 1.0, 'sq': 1.0, 'rq': 1.0, 'tp': 0, 'fp': 0, 'fn': 0}
    if len(pred_instances) == 0:
        return {'pq': 0.0, 'sq': 0.0, 'rq': 0.0, 'tp': 0, 'fp': 0, 'fn': len(gt_instances)}
    if len(gt_instances) == 0:
        return {'pq': 0.0, 'sq': 0.0, 'rq': 0.0, 'tp': 0, 'fp': len(pred_instances), 'fn': 0}

    matched_gt = set()
    iou_sum = 0.0
    tp = 0
    fp = 0
    for pred in pred_instances:
        best_iou = 0.0
        best_gt_idx = -1
        for gt_idx, gt in enumerate(gt_instances):
            if gt_idx in matched_gt: continue
            inter = np.logical_and(pred, gt).sum()
            union = np.logical_or(pred, gt).sum()
            if union > 0:
                iou = inter / union
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
        if best_iou > 0.5:
            tp += 1
            iou_sum += best_iou
            matched_gt.add(best_gt_idx)
        else:
            fp += 1
    fn = len(gt_instances) - tp
    sq = iou_sum / tp if tp > 0 else 0.0
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    return {'pq': sq * rq, 'sq': sq, 'rq': rq, 'tp': tp, 'fp': fp, 'fn': fn}

### 7. Instance Separation: Marker-Controlled Watershed & Official Pycocotools RLE Encoder

In [ ]:
def separate_instances(binary_mask: np.ndarray, min_area=100) -> list:
    dist_transform = cv2.distanceTransform(binary_mask.astype(np.uint8), cv2.DIST_L2, 5)
    max_d = float(dist_transform.max())
    if max_d <= 0.0:
        return []
    _, markers_mask = cv2.threshold(dist_transform, 0.4 * max_d, 255, 0)
    markers_mask = np.uint8(markers_mask)
    _, markers = cv2.connectedComponents(markers_mask)
    labeled = watershed(-dist_transform, markers, mask=binary_mask)
    instances = []
    for region in regionprops(labeled):
        if region.area >= min_area:
            instances.append((labeled == region.label).astype(np.uint8))
    return instances

def mask_to_pycoco_rle(mask: np.ndarray) -> str:
    from pycocotools import mask as mask_util
    encoded = mask_util.encode(np.asfortranarray((mask > 0).astype(np.uint8)))
    counts = encoded['counts']
    return counts.decode('utf-8') if isinstance(counts, bytes) else str(counts)

def pycoco_rle_to_mask(coco_rle: str, shape=(2048, 2048)) -> np.ndarray:
    from pycocotools import mask as mask_util
    return np.asfortranarray(mask_util.decode({'size': list(shape), 'counts': str(coco_rle).strip().encode('utf-8')}).astype(np.uint8))

### 8. Inference Pipeline on Blind Test Images & Kaggle Submission Export

In [ ]:
def load_solar_grayscale_image(img_path, target_shape=(2048, 2048)):
    ext = os.path.splitext(img_path)[1].lower()
    if ext in {".fits", ".fit"}:
        from astropy.io import fits
        with fits.open(img_path, memmap=False) as hdul:
            data = next((np.asarray(h.data, dtype=np.float32) for h in hdul if getattr(h, "data", None) is not None and np.ndim(h.data) >= 2), None)
            if data is None:
                raise ValueError(f"FITS file {img_path} contains no 2D image")
            data = np.squeeze(data)
            vmin, vmax = float(np.percentile(data[np.isfinite(data)], 0.5)), float(np.percentile(data[np.isfinite(data)], 99.5))
            img = np.zeros(data.shape, dtype=np.uint8) if vmax <= vmin else (np.clip((data - vmin) / (vmax - vmin), 0.0, 1.0) * 255.0).astype(np.uint8)
    else:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Failed to read image file {img_path}")
    if img.shape != target_shape:
        img = cv2.resize(img, (target_shape[1], target_shape[0]), interpolation=cv2.INTER_AREA)
    return img

def generate_submission(test_dir="/kaggle/input/filament-segmentation-2026/test", model=None, output_csv="submission.csv", image_id_map=None, sample_submission_path="/kaggle/input/filament-segmentation-2026/sample_submission.csv"):
    if not os.path.exists(test_dir):
        print(f"Test directory {test_dir} not found. Please mount competition dataset in Kaggle Notebooks.")
        return None
    valid_ids = None
    if sample_submission_path and os.path.exists(sample_submission_path):
        sdf = pd.read_csv(sample_submission_path, dtype=str)
        if "image_id" in sdf.columns:
            valid_ids = set(sdf["image_id"].dropna().astype(str))
        elif "filament_id" in sdf.columns:
            valid_ids = {str(x).split("_", 1)[0] for x in sdf["filament_id"].dropna()}
    files = sorted([f for f in os.listdir(test_dir) if f.lower().endswith((".png", ".fits", ".fit", ".jpg", ".jpeg", ".tif", ".tiff"))])
    print(f"Processing {len(files)} test images for submission...")
    submission_rows = []
    seen_ids = set()
    predicted_images = 0
    zero_pred_images = 0
    for f in files:
        img_path = os.path.join(test_dir, f)
        raw = load_solar_grayscale_image(img_path, target_shape=(2048, 2048))
        instances = segment_filament_instances(raw)
        base_id = image_id_map[f] if (image_id_map and f in image_id_map) else os.path.splitext(f)[0]
        if valid_ids is not None and base_id not in valid_ids:
            raise ValueError(f"Image ID {base_id} not found in official sample_submission registry")
        if instances:
            predicted_images += 1
        else:
            zero_pred_images += 1
        for idx, inst in enumerate(instances):
            fil_id = f"{base_id}_{idx + 1}"
            if fil_id in seen_ids:
                raise ValueError(f"Duplicate filament_id detected: {fil_id}")
            seen_ids.add(fil_id)
            submission_rows.append({"filament_id": fil_id, "segmentation_rle": mask_to_pycoco_rle(inst)})
    df = pd.DataFrame(submission_rows, columns=["filament_id", "segmentation_rle"])
    df.to_csv(output_csv, index=False, encoding="utf-8", lineterminator="\n")
    recheck = pd.read_csv(output_csv, dtype=str)
    assert list(recheck.columns) == ["filament_id", "segmentation_rle"]
    for rle in recheck["segmentation_rle"]:
        m = pycoco_rle_to_mask(rle, shape=(2048, 2048))
        assert m.shape == (2048, 2048) and mask_to_pycoco_rle(m) == rle
    print(f"SUCCESS: Processed {len(files)} test images ({predicted_images} with detections, {zero_pred_images} empty) -> {len(recheck)} rows verified in {output_csv}")
    return recheck

print("End-to-end pipeline ready for execution.")